In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, classification_report, accuracy_score, confusion_matrix
import joblib, os

DATA_PATH = "/mnt/data/jumlah-siswa-mengulang-menurut-tingkat-tiap-propinsi-kota-makassar-sd-2024.xlsx"
OUTPUT_DIR = "/mnt/data/mengulang_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Paths set. Output dir:', OUTPUT_DIR)

Paths set. Output dir: /mnt/data/mengulang_output


In [4]:
DATA_PATH = r"C:\Users\k\Documents\MID\jumlah-siswa-mengulang-menurut-tingkat-tiap-propinsi-kota-makassar-sd-2024.xlsx"

df = pd.read_excel(DATA_PATH)
df.head()


,Jumlah Siswa Mengulang Menurut Tingkat Tiap Provinsi (Kota Makassar) Tahun 2024/2025 SD,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Kecamatan,Tingkat - I,Tingkat - II,Tingkat - III,Tingkat - IV,Tingkat - V,Tingkat - VI,Jumlah,Status
2,Kec. Mariso,2,7,4,5,2,0,20,Negeri
3,Kec. Mamajang,2,2,0,0,2,1,7,Negeri
4,Kec. Tamalate,4,1,1,4,20,0,30,Negeri


In [5]:
for c in df.columns:
    if df[c].dtype.kind in "iufc":
        df[c] = df[c].fillna(0)
    else:
        df[c] = df[c].fillna("").astype(str).str.strip()

In [7]:
# Identifikasi kolom tingkat (pengganti Kelas_1..Kelas_6)
kelas_cols = [c for c in df.columns if "tingkat" in c.lower()]

# Tentukan target Total_Mengulang
if "Total_Mengulang" in df.columns:
    target_col = "Total_Mengulang"
elif "Jumlah" in df.columns:
    target_col = "Jumlah"
    print("Kolom 'Jumlah' digunakan sebagai Total_Mengulang.")
elif len(kelas_cols) > 0:
    df["Total_Mengulang"] = df[kelas_cols].sum(axis=1)
    target_col = "Total_Mengulang"
    print("Total_Mengulang dibuat otomatis dari kolom Tingkat.")
else:
    raise ValueError("Tidak ada kolom untuk menentukan Total_Mengulang.")


Total_Mengulang dibuat otomatis dari kolom Tingkat.


In [10]:
# Paksa konversi ke numerik
df["Total_Mengulang"] = pd.to_numeric(df["Total_Mengulang"], errors="coerce")

df["Total_Mengulang"] = df["Total_Mengulang"].fillna(0)

def risk_label(x):
    if x <= 5:
        return "Rendah"
    elif x <= 15:
        return "Sedang"
    else:
        return "Tinggi"

df["Risk_Category"] = df["Total_Mengulang"].apply(risk_label)


In [ ]:
# Remove kelas columns from features because mereka menjumlah ke target
features_all = df.columns.tolist()
exclude = set(["Nama_Sekolah","Total_Mengulang","Risk_Category"]) | set(kelas_cols)
feature_candidates = [c for c in features_all if c not in exclude]

print("Feature candidates (after excluding kelas & target):", feature_candidates)

numeric_feats = [c for c in feature_candidates if df[c].dtype.kind in "iufc"]
cat_feats = [c for c in feature_candidates if df[c].dtype == object]

print("Numeric features to use:", numeric_feats)
print("Categorical features to use:", cat_feats)

# If there are numeric features besides kelas, we'll build a regression model for Total_Mengulang.
do_regression = len(numeric_feats) > 0

# Prepare preprocessing
# For categorical: OneHotEncoder (handle_unknown)
# For numeric: MinMaxScaler

# Build X, y for regression (if possible)
if do_regression:
    X_reg = pd.DataFrame()
    if numeric_feats:
        X_reg[numeric_feats] = df[numeric_feats].astype(float)
    if cat_feats:
        # one-hot encode categorical manually here (so the final X is numpy for models)
        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
        cat_enc = ohe.fit_transform(df[cat_feats])
        cat_cols = []
        for i, col in enumerate(cat_feats):
            cats = ohe.categories_[i]
            cat_cols += [f"{col}__{v}" for v in cats]
        X_reg = pd.concat([X_reg.reset_index(drop=True), pd.DataFrame(cat_enc, columns=cat_cols)], axis=1)
    y_reg = df["Total_Mengulang"].astype(float).values

    # scale numeric columns (only numeric part)
    scaler = MinMaxScaler()
    if numeric_feats:
        X_reg[numeric_feats] = scaler.fit_transform(X_reg[numeric_feats])

    # train-test split
    Xtr, Xte, ytr, yte = train_test_split(X_reg.values, y_reg, test_size=0.2, random_state=42)

    # models
    lr = LinearRegression()
    rf_reg = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)

    # fit
    lr.fit(Xtr, ytr)
    rf_reg.fit(Xtr, ytr)

    # eval
    for name, model in [("LinearRegression", lr), ("RandomForestRegressor", rf_reg)]:
        yp = model.predict(Xte)
        print(name)
        print(" MAE:", mean_absolute_error(yte, yp))
        print(" MSE:", mean_squared_error(yte, yp))
        print(" R2 :", r2_score(yte, yp))
        print("---")

    # choose best
    best_reg = rf_reg if r2_score(yte, rf_reg.predict(Xte)) >= r2_score(yte, lr.predict(Xte)) else lr
    # save artifacts
    joblib.dump(best_reg, os.path.join(OUTPUT_DIR, "best_reg_model.pkl"))
    if 'scaler' in locals():
        joblib.dump(scaler, os.path.join(OUTPUT_DIR, "scaler_reg.pkl"))
    if 'ohe' in locals():
        joblib.dump(ohe, os.path.join(OUTPUT_DIR, "ohe_reg.pkl"))

else:
    print("Tidak ada fitur numerik tambahan — melewatkan regression. Kita akan melakukan klasifikasi Risk_Category berbasis fitur yang ada (mis. Kecamatan).")

Feature candidates (after excluding kelas & target): ['Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8']
Numeric features to use: []
Categorical features to use: ['Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8']
Tidak ada fitur numerik tambahan — melewatkan regression. Kita akan melakukan klasifikasi Risk_Category berbasis fitur yang ada (mis. Kecamatan).


In [12]:
# Use categorical features (like Kecamatan) and any numeric features (if exist)
clf_feats = []
if cat_feats:
    clf_feats += cat_feats
if numeric_feats:
    clf_feats += numeric_feats

if len(clf_feats) == 0:
    # fallback: if nothing left, use Kecamatan if present in original columns
    if "Kecamatan" in df.columns:
        clf_feats = ["Kecamatan"]
    else:
        raise ValueError("Tidak ada fitur untuk klasifikasi. Pastikan dataset memiliki Kecamatan atau fitur lainnya.")

print("Features for classification:", clf_feats)

# build X_clf
Xc_num = df[[c for c in clf_feats if df[c].dtype.kind in "iufc"]].astype(float) if any(df[c].dtype.kind in "iufc" for c in clf_feats) else pd.DataFrame()
Xc_cat = df[[c for c in clf_feats if df[c].dtype == object]].copy() if any(df[c].dtype == object for c in clf_feats) else pd.DataFrame()

# encode categorical
if not Xc_cat.empty:
    ohe_clf = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    cat_enc_clf = ohe_clf.fit_transform(Xc_cat)
    cat_cols_clf = []
    for i, col in enumerate(Xc_cat.columns):
        cats = ohe_clf.categories_[i]
        cat_cols_clf += [f"{col}__{v}" for v in cats]
    Xc = pd.concat([Xc_num.reset_index(drop=True), pd.DataFrame(cat_enc_clf, columns=cat_cols_clf)], axis=1)
else:
    Xc = Xc_num.copy()

# scale numeric part
if not Xc_num.empty:
    scaler_clf = MinMaxScaler()
    Xc[Xc_num.columns.tolist()] = scaler_clf.fit_transform(Xc[Xc_num.columns.tolist()])

y_clf = df["Risk_Category"].values

# train-test split
Xctr, Xcte, yctr, ycte = train_test_split(Xc.values, y_clf, test_size=0.2, random_state=42)

# classifier
clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
clf.fit(Xctr, yctr)
y_pred_clf = clf.predict(Xcte)

print("Classification results (Risk_Category):")
print(" Accuracy:", accuracy_score(ycte, y_pred_clf))
print(classification_report(ycte, y_pred_clf))
print("Confusion matrix:\n", confusion_matrix(ycte, y_pred_clf))

# save classifier and artifacts
joblib.dump(clf, os.path.join(OUTPUT_DIR, "clf_risk.pkl"))
if 'ohe_clf' in locals():
    joblib.dump(ohe_clf, os.path.join(OUTPUT_DIR, "ohe_clf.pkl"))
if 'scaler_clf' in locals():
    joblib.dump(scaler_clf, os.path.join(OUTPUT_DIR, "scaler_clf.pkl"))

print("Saved artifacts in", OUTPUT_DIR)

Features for classification: ['Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8']
Classification results (Risk_Category):
 Accuracy: 1.0
              precision    recall  f1-score   support

      Rendah       1.00      1.00      1.00         7

    accuracy                           1.00         7
   macro avg       1.00      1.00      1.00         7
weighted avg       1.00      1.00      1.00         7

Confusion matrix:
 [[7]]


c:\Users\k\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


Saved artifacts in /mnt/data/mengulang_output


In [13]:
if "Kecamatan" in df.columns:
    by_kec = df.groupby("Kecamatan")["Total_Mengulang"].sum().sort_values(ascending=False)
    topk = by_kec.head(15)
    plt.figure(figsize=(10,5))
    plt.bar(topk.index, topk.values)
    plt.xticks(rotation=45, ha="right")
    plt.title("Top 15 Kecamatan by Total Mengulang")
    plt.tight_layout()
    plt.show()